In [2]:
!pip install scikit-learn xgboost catboost joblib pandas numpy

  Using cached scikit_learn-1.7.2-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached scipy-1.16.3-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached plotly-6.4.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached narwhals-2.11.0-py3-none-any.whl.metadata (11 kB)
Using cached scikit_learn-1.7.2-cp313-cp313-win_amd64.whl (8.7 MB)
   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   - -------------------------------------- 2.9/72.0 MB 17.3 MB/s eta 0:00:04
   ---- ----------------------------------- 7.9/72.0 MB 20.1 MB/s eta 0:00:04
   ----- ---------------------------------- 10.7/72.0 MB 20.8 MB/s eta 0:00:03
   -------- ------------------------------- 14.9/72.0 MB 18.8 MB/s eta 0:00:04
   --------- ------------------------------ 17.8/72.0 MB 18.5 MB/s eta 0:00:03
   ----------- ---------------------------- 21.0/72.0 MB 17.7 


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os

In [4]:
# Create models directory if needed
os.makedirs('models', exist_ok=True)

In [6]:
# Load cleaned data (adjusted path to match project structure)
df = pd.read_csv('C:\Solar Prediction\Data\Data\clean_solar_data.csv', index_col='datetime', parse_dates=True)

<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
C:\Users\altam\AppData\Local\Temp\ipykernel_17500\4287778715.py:2: SyntaxWarning: invalid escape sequence '\S'
  df = pd.read_csv('C:\Solar Prediction\Data\Data\clean_solar_data.csv', index_col='datetime', parse_dates=True)


In [7]:
# Feature engineering: Cyclical features for time (using datetime index)
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
df['day_sin'] = np.sin(2 * np.pi * df.index.dayofyear / 365)
df['month_sin'] = np.sin(2 * np.pi * df.index.month / 12)

In [8]:
# Define features and target (predict AC_POWER from weather/time features)
# Categorical: PLANT_ID (one-hot encode)
categorical_features = ['PLANT_ID']
numerical_features = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION',
                      'hour_sin', 'hour_cos', 'day_sin', 'month_sin']

In [9]:
X = df[numerical_features + categorical_features]
y = df['AC_POWER']

In [10]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
# Preprocessing pipeline: Scale numerical, one-hot encode categorical
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ])

In [12]:
# Apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [14]:
# Train models
models = {
    'Ridge': Ridge(),
    'SVR': SVR(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'CATBoost': CatBoostRegressor(verbose=0, random_state=42),
    'Neural Network': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    rms = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'RMS': rms, 'MAE': mae, 'R²': r2}
    print(f"{name} - RMS: {rms:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

Ridge - RMS: 2378.3703, MAE: 1511.3190, R²: 0.9023
SVR - RMS: 8675.3308, MAE: 5306.5350, R²: -0.2996
Random Forest - RMS: 1140.8562, MAE: 392.2328, R²: 0.9775
XGBoost - RMS: 1160.4876, MAE: 416.4946, R²: 0.9767
CATBoost - RMS: 1135.6792, MAE: 411.6737, R²: 0.9777
Neural Network - RMS: 1463.8221, MAE: 554.3339, R²: 0.9630


c:\Solar Prediction\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [15]:
# Print results summary
print("\nModel Performance Summary:")
results_df = pd.DataFrame(results).T
print(results_df)


Model Performance Summary:
                        RMS          MAE        R²
Ridge           2378.370337  1511.319008  0.902323
SVR             8675.330751  5306.534984 -0.299591
Random Forest   1140.856194   392.232764  0.977525
XGBoost         1160.487558   416.494629  0.976745
CATBoost        1135.679158   411.673745  0.977729
Neural Network  1463.822094   554.333950  0.962999


In [16]:
# Best model: Select based on highest R² (adapt from results; example assumes Neural Network performs well)
best_model_name = max(results, key=lambda k: results[k]['R²'])
best_model = models[best_model_name]
print(f"\nBest model: {best_model_name} (R² = {results[best_model_name]['R²']:.4f})")


Best model: CATBoost (R² = 0.9777)


In [17]:
# Save full pipeline: preprocessor + best model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', best_model)
])
joblib.dump(pipeline, 'models/model_pipeline.pkl')
print("Pipeline saved to models/model_pipeline.pkl")

Pipeline saved to models/model_pipeline.pkl
